**Features Extraction**

In [1]:
import os
import librosa
import pandas as pd
import numpy as np
from scipy.io import wavfile
import matplotlib.pyplot as plt
import matplotlib
import cv2
import warnings
matplotlib.use('Agg')

data_path_Train = pd.read_csv("CSVs\\ravdess_train.csv")
data_path_Test = pd.read_csv("CSVs\\ravdess_test.csv")

In [ ]:
def extract_features(ef_data: np.ndarray, ef_sample_rate: int) -> np.ndarray:

    ef_features = np.array([])

    # ZCR
    zcr = np.mean(librosa.feature.zero_crossing_rate(ef_data).T, axis=0)
    ef_features = np.hstack((ef_features, zcr))

    # Chroma STFT
    stft = librosa.stft(ef_data)
    chroma_stft = np.mean(
        librosa.feature.chroma_stft(S=np.abs(stft), sr=ef_sample_rate).T, axis=0
    )
    ef_features = np.hstack((ef_features, chroma_stft))

    # MFCC
    mfcc = np.mean(librosa.feature.mfcc(y=ef_data, sr=ef_sample_rate).T, axis=0)
    ef_features = np.hstack((ef_features, mfcc))

    # Root Mean Square Value
    rmse = np.mean(librosa.feature.rms(y=ef_data).T, axis=0)
    ef_features = np.hstack((ef_features, rmse))

    # Mel Spectrogram
    mel = np.mean(
        librosa.feature.melspectrogram(y=ef_data, sr=ef_sample_rate).T, axis=0
    )
    ef_features = np.hstack((ef_features, mel))

    return ef_features


def get_features(gf_path: str) -> np.ndarray:
    data, sample_rate = librosa.load(gf_path, sr=None)

    sample_rate = int(sample_rate)
    features = extract_features(data, sample_rate)

    return features


def prepare_audios_augmented(pa_df: pd.DataFrame, pa_name: str):
    data, labels = [], []
    total = len(pa_df) * 5

    for _, row in pa_df.iterrows():
        try:
            audio, sr = librosa.load(row["path"], sr=None)

            augmented = [
                audio,
                audio + 0.005 * np.random.randn(len(audio)),
                librosa.effects.pitch_shift(audio, sr=sr, n_steps=2),
                librosa.effects.pitch_shift(audio, sr=sr, n_steps=-2),
                librosa.effects.time_stretch(audio, rate=0.9),
            ]

            for aug in augmented:
                data.append(extract_features(aug, int(sr)))
                labels.append(row["emotion"])
                print(f"\r{pa_name} - Saved {len(data)}/{total}", end="", flush=True)


        except Exception as e:
            print(f"\nError: {row['path']} → {e}")

    os.makedirs("features", exist_ok=True)  # ← thêm
    np.save(f"features\\{pa_name}_data.npy", np.array(data))
    np.save(f"features\\{pa_name}_labels.npy", np.array(labels))
    print(f"\n{pa_name} Done — {len(data)} samples")


def prepare_audios(pa_df: pd.DataFrame, pa_name: str):
    data = []
    labels = []
    total = len(pa_df)

    for _, row in pa_df.iterrows():
        try:
            features = get_features(row["path"])
            data.append(features)
            labels.append(row["emotion"])
            print(f"{pa_name} - Done! Saved {len(data)}/{total}", end="\r", flush=True)

        except Exception as e:
            print(f"Error processing {row['path']}: {e}")

    data = np.array(data)
    labels = np.array(labels)

    os.makedirs("features", exist_ok=True)

    np.save(f"features\\{pa_name}_data.npy", data)
    print()
    np.save(f"features\\{pa_name}_labels.npy", labels)


prepare_audios_augmented(data_path_Train, "train")
prepare_audios(data_path_Test, "test")

train - Saved 5760/5760train Done — 5760 samples
test - Done! Saved 288/288


**Create spectrogram image**

In [5]:
location = "features\\images\\"


def graph_spectrogram(gs_audio, gs_filename: str, sr: int = None): # type: ignore
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        # Nhận cả path lẫn array
        if isinstance(gs_audio, str):
            audio, sr = librosa.load(gs_audio, sr=None) # type: ignore
        else:
            audio = gs_audio  # array trực tiếp

        mel = librosa.feature.melspectrogram(y=audio, sr=sr)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        fig, ax = plt.subplots(1)
        fig.set_size_inches(2.24, 2.24)
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
        ax.axis("off")

        librosa.display.specshow(mel_db, ax=ax, cmap="inferno")

        plt.savefig(gs_filename, dpi="figure")
        plt.close(fig)


def prepare_images_augmented(pi_df: pd.DataFrame, pi_Name: str):
    os.makedirs(location, exist_ok=True)
    os.makedirs("features", exist_ok=True)

    data = []
    labels = []
    total = len(pi_df) * 5
    counter = 1

    for _, audio in pi_df.iterrows():
        try:
            original, sr = librosa.load(audio["path"], sr=None)

            augmented = [
                original,
                original + 0.005 * np.random.randn(len(original)),
                librosa.effects.pitch_shift(original, sr=sr, n_steps=2),
                librosa.effects.pitch_shift(original, sr=sr, n_steps=-2),
                librosa.effects.time_stretch(original, rate=0.9),
            ]

            for aug in augmented:
                filename = f"{location}{pi_Name}_{counter}.png"
                graph_spectrogram(aug, filename, sr=int(sr))  # array trực tiếp

                img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue

                img = cv2.resize(img, (224, 224))
                data.append(img)
                labels.append(audio["emotion"])
                print(f"\r{pi_Name} - Saved {len(data)}/{total}", end="", flush=True)
                counter += 1

        except Exception as e:
            print(f"\nError: {audio['path']} → {e}")

    data = np.array(data, dtype=float)
    labels = np.array(labels)

    np.save(f"features\\{pi_Name}_images.npy", data)
    np.save(f"features\\{pi_Name}_image_labels.npy", labels)
    print(f"\n{pi_Name} Done — {len(data)} samples")


def prepare_images(pi_df: pd.DataFrame, pi_Name: str):
    os.makedirs(location, exist_ok=True)
    os.makedirs("features", exist_ok=True)

    data = []
    labels = []
    counter = 1

    for _, audio in pi_df.iterrows():
        try:
            filename = f"{location}{pi_Name}_{counter}.png"
            graph_spectrogram(audio["path"], filename)
            img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"\nError reading image {filename}")
                continue

            img = cv2.resize(img, (224, 224))
            data.append(img)
            labels.append(audio["emotion"])
            print(f"\r{pi_Name} - Saved {len(data)}/{len(pi_df)}", end="", flush=True)
            counter += 1

        except Exception as e:
            print(f"\nError: {audio['path']} → {e}")

    data = np.array(data, dtype=float)
    labels = np.array(labels)

    np.save(f"features\\{pi_Name}_images.npy", data)
    np.save(f"features\\{pi_Name}_image_labels.npy", labels)
    print(f"\n{pi_Name} Done — {len(data)} samples")


# Train augment × 5, Test giữ nguyên
prepare_images_augmented(data_path_Train, "train")
prepare_images(data_path_Test, "test")

train - Saved 5760/5760
train Done — 5760 samples
test - Saved 288/288
test Done — 288 samples
